# Certify the fault distance of circuits

Certifying a circuit's fault distance by brute-force enumeration requires searching combinations of faults, whose number grows rapidly with the circuit size and the distance to certify.
Instead, we map a circuit's detector error model to an auxiliary CSS code and leverage the Brouwer-Zimmermann algorithm for computing code distance to certify the fault distance of the circuit.

The examples below certify the distance of a surface-code memory experiment and a flagged Steane-state preparation circuit.
Unlike the preceding notebooks, these calculations are exact searches rather than Monte Carlo estimates of logical error rates.

## Setup

Run the next cell once in a fresh notebook environment.
IPython's `%pip` magic installs into the active kernel, so the following imports use the same environment.

In [1]:
%%capture
%pip install qldpc

In [2]:
import itertools

import galois
import numpy as np
import stim

from qldpc import circuits, codes, decoders

## Turn circuit faults into a CSS code

`DetectorErrorModelArrays` provides two binary matrices: `detectors` maps fault mechanisms to detector flips, and `observables` maps the same mechanisms to logical-observable flips.
Let these matrices be $D$ and $L$, respectively.
A binary vector $e$ selects fault mechanisms, and its Hamming weight $|e|$ counts them.
An undetected logical fault satisfies $De=0$ and $Le\ne0$, so the distance we want is

$$
d_{\mathrm{fault}}=\min_{De=0,\;Le\ne0}|e|.
$$

Each column represents one DEM fault mechanism with unit cost.

Imagine one artificial qubit per fault mechanism, with $e$ represented by a Z-type operator on those qubits.
Set the auxiliary code's X-check matrix to $H_X=D$.
Its Z-logical operators must then have zero detector syndrome, $De=0$.

Next set $H_Z$ to a basis for the common null space of $D$ and $L$.
These Z stabilizers have zero detector syndrome and commute with the auxiliary X-type operators represented by the rows of $L$.
They are precisely the undetected fault patterns with no observable effect, which we declare trivial.
The remaining nontrivial Z-logical operators have zero syndrome but flip an observable.

The Z distance of this auxiliary code is therefore the circuit fault distance.
The artificial Z label does not restrict the underlying circuit faults to physical Z errors.

In [3]:
def get_fault_code(dem_arrays: decoders.DetectorErrorModelArrays) -> codes.CSSCode:
    """Construct a CSS code whose Z distance is the undetected logical fault distance."""
    detectors = galois.GF2(dem_arrays.detector_flip_matrix.toarray())
    observables = galois.GF2(dem_arrays.observable_flip_matrix.toarray())
    harmless_faults = np.vstack([detectors, observables]).null_space()
    return codes.CSSCode(detectors, harmless_faults)

## Certify a memory experiment

We use a distance-5 surface code with five rounds of syndrome extraction, testing X- and Z-basis memory separately.
Only the QEC cycle is noisy; initialization and readout are noiseless.
The depolarizing noise model applies noise to active operations and leaves idle qubits noiseless.
Constructing all rounds together gives syndrome-difference detectors between adjacent rounds.

`get_distance("Z")` computes the exact Z distance of each auxiliary code.
We use Stim's graphlike search as a cross-check for this surface-code example because each fault in its syndrome-extraction circuit flips at most two detectors.
General memory experiments can contain non-graphlike faults, for which Stim's graphlike search need not identify the true fault distance.
The Brouwer-Zimmermann calculation does not require graphlike faults.

In [4]:
code = codes.SurfaceCode(5)
noise_model = circuits.DepolarizingNoiseModel(1e-3)

for basis in ("X", "Z"):
    # one logical memory cycle = code.get_distance() rounds of syndrome extraction
    # you can replace logical_memory_cycle with another memory circuit
    initialization, logical_memory_cycle, readout, *_ = circuits.get_memory_experiment_parts(
        code, basis=basis, num_rounds=code.get_distance()
    )

    noisy_circuit = initialization + noise_model.noisy_circuit(logical_memory_cycle) + readout
    dem_arrays = decoders.DetectorErrorModelArrays(noisy_circuit)
    fault_code = get_fault_code(dem_arrays)

    fault_distance = fault_code.get_distance("Z")
    graphlike_distance = len(noisy_circuit.shortest_graphlike_error())
    assert fault_distance == graphlike_distance
    print(
        f"{basis}-basis memory: fault distance {fault_distance}, "
        f"Stim graphlike distance {graphlike_distance}"
    )

X-basis memory: fault distance 5, Stim graphlike distance 5


Z-basis memory: fault distance 5, Stim graphlike distance 5


Both memory bases use the auxiliary code's Z distance: this Z label describes fault vectors, not the basis of the memory experiment.

## Certify a flagged state-preparation circuit

We now prepare the Steane code in $|0_L\rangle$, using the flagged circuit from the [state-preparation notebook](5_state_preparation.ipynb).
Every measurement in the preparation circuit is post-selected: a nonzero outcome rejects the attempt and restarts preparation.
Here the only such measurement is the final X-basis measurement of qubit 7.
The flag checks a logical-Z representative, so it can reveal logical-X faults in the preparation.

In [5]:
code = codes.SteaneCode()

state_prep_circuit = stim.Circuit("""
    # non-fault-tolerant state preparation
    H 0 2 4
    CX 0 3 2 1 4 5
    CX 0 1 2 6 4 3
    CX 2 5 3 6
    # flag errors in a logical Z representative
    H 7
    CZ 7 1 7 3 7 5
    MX 7
""")

`get_state_prep_diagnostic_circuit` adds a flag detector for the preparation measurement, noiseless final stabilizer measurements, and a noiseless logical-Z measurement.
The diagnostic operations remain noiseless when we apply the noise model.

For the undetected logical fault distance, we require both the flag and the final stabilizer syndrome to be zero while the logical-Z observable flips.
The final stabilizer detectors are diagnostics, not additional rejection tests performed by the original preparation protocol.

In [6]:
diagnostic_circuit, _ = circuits.get_state_prep_diagnostic_circuit(
    code,
    state_prep_circuit,
    add_flags=True,
    observables=code.get_logical_ops("Z", symplectic=True),
)
noisy_circuit = noise_model.noisy_circuit(diagnostic_circuit)
dem_arrays = decoders.DetectorErrorModelArrays(noisy_circuit)
fault_code = get_fault_code(dem_arrays)

fault_distance = fault_code.get_distance("Z")
print(f"State-preparation undetected logical fault distance: {fault_distance}")

State-preparation undetected logical fault distance: 3


A three-fault witness is an X error on each of qubits 1, 3, and 5 immediately after its verification CZ gate.
Each fault escapes the flag, and together they leave the logical operator $X_1X_3X_5$, which has zero stabilizer syndrome and flips the logical-Z observable.
The exact distance calculation also certifies that no one- or two-fault pattern has all these properties.

## Account for Z faults in a prepared zero state (WORK IN PROGRES)

To characterize Z faults before coupling the prepared block to another block, we compute their effective weight on the prepared Steane state.
This calculation uses the seven physical data qubits.

The prepared state is stabilized by the code's Z checks and by its logical Z, $\overline Z=Z_1Z_3Z_5$.
Multiplying an output error by any of these operators leaves its effect on $|0_L\rangle$ unchanged, including through subsequent interactions.

An $X_7$ fault on the flag ancilla propagates Z errors through the remaining CZ gates without flipping its final X-basis measurement.
The resulting data errors illustrate why physical weight alone can overstate the damage:

| Position of the ancilla X fault | Data Z error | Equivalent effect on the prepared state |
| --- | --- | --- |
| Before all three CZ gates | $Z_1Z_3Z_5$ | Identity |
| After the first CZ gate | $Z_3Z_5$ | $Z_1$ |
| After the second CZ gate | $Z_5$ | $Z_5$ |

For example, $Z_3Z_5|0_L\rangle=Z_1\overline Z|0_L\rangle=Z_1|0_L\rangle$.
The apparent two-qubit error is equivalent to a single-qubit Z error.

The rows of the following generator matrix span all harmless pure-Z operators on the prepared state.
The distance from an output Z-error vector to this classical code is its minimum effective weight after multiplying by those operators.

In [7]:
harmless_z = codes.ClassicalCode.from_generator(
    np.vstack([code.matrix_z, code.get_logical_ops("Z")])
)

for support, expected_weight in (((1, 3, 5), 0), ((3, 5), 1), ((5,), 1)):
    error = code.field.Zeros(len(code))
    error[list(support)] = 1
    effective_weight = harmless_z.get_distance(vector=error)
    assert effective_weight == expected_weight
    print(f"Z error on {support}: effective weight {effective_weight}")

Z error on (1, 3, 5): effective weight 0
Z error on (3, 5): effective weight 1
Z error on (5,): effective weight 1


For this Steane zero state, every pure-Z error is equivalent to identity or a single-qubit Z error.
We can verify this by checking the $2^7=128$ output Z patterns, a small calculation distinct from enumerating the much larger circuit fault space.
This is a special property of the known Steane zero state, not a guarantee for arbitrary CSS codes or arbitrary logical inputs.

In [8]:
max_effective_z_weight = max(
    harmless_z.get_distance(vector=code.field(bits))
    for bits in itertools.product((0, 1), repeat=len(code))
)
assert max_effective_z_weight == 1
print(f"Maximum effective weight of any output Z error: {max_effective_z_weight}")

Maximum effective weight of any output Z error: 1


For this prepared Steane state, every pure-Z error has a representative supported on at most one data qubit.
Such a representative propagates to at most one qubit per block under a transversal coupling.
Subsequent syndrome extraction and correction must account for these residual errors.

Brouwer-Zimmermann avoids an exhaustive codeword search when its bounds meet early, but exact distance remains a combinatorial calculation that can be expensive for larger circuits.